# SoW submission: inspect, reproduce, demonstrate

Default execution makes **no model API calls**. This wrapper reads the submitted frozen v4.1 project; it was prepared after the experiment. Use the repository ZIP, not an older results ZIP. No local Python installation is needed. Steps 4 and 5 are optional and disabled by default.

## 1. Upload and unpack the repository ZIP
Upload `SoW_GitHub_Submission.zip` or GitHub’s Download ZIP. Existing extraction folders are not overwritten.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, io, tempfile, os, json, hashlib, sys

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one repository ZIP.")
archive_bytes = next(iter(uploaded.values()))
extract_root = Path(tempfile.mkdtemp(prefix="sow_submission_", dir="/content"))
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    if sum(item.file_size for item in archive.infolist()) > 100_000_000:
        raise ValueError("Archive exceeds this notebook's 100 MB limit.")
    for item in archive.infolist():
        target = (extract_root / item.filename).resolve()
        if not target.is_relative_to(extract_root.resolve()):
            raise ValueError("Unsafe archive path.")
        if (item.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError("Archive symlinks are not accepted.")
    archive.extractall(extract_root)
roots = [p.parent for p in extract_root.rglob("freeze_manifest.json")
         if (p.parent / "sow.py").is_file()]
if len(roots) != 1:
    raise ValueError("Expected exactly one frozen SoW project.")
PROJECT = roots[0]
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print("Project:", PROJECT)

## 2. Verify original frozen files
This checks file integrity, not accuracy or model behaviour.

In [ ]:
manifest = json.loads(Path("freeze_manifest.json").read_text())
mismatches = []
for relative_path, expected in manifest["files"].items():
    path = PROJECT / relative_path
    actual = hashlib.sha256(path.read_bytes()).hexdigest() if path.is_file() else None
    if actual != expected:
        mismatches.append(relative_path)
if mismatches:
    raise ValueError("Frozen files differ: " + ", ".join(mismatches))
print("All", len(manifest["files"]), "frozen files match the original manifest.")

## 3. Inspect final results and one saved case
No API key is needed. The summaries are preserved in their original language; the README explains the results in English.

In [ ]:
from IPython.display import display, Markdown
display(Markdown(Path("README.md").read_text()))
for split in ["dev", "holdout"]:
    metrics = json.loads(Path(f"reports/{split}/metrics.json").read_text())
    print(split, json.dumps(metrics["summary"], indent=2, ensure_ascii=False))
record = json.loads(Path("saved_dev_ai/DEV01.json").read_text())
print("Saved DEV01 status:", record.get("status"))
print(json.dumps(record.get("validated_output"), ensure_ascii=False, indent=2))

## 4. Optional: reproduce baseline and scoring without model calls

Outputs go to `reproduction/`, leaving the submitted reports intact. Existing manual decisions are reused. This does not independently validate them. The frozen numeric-coverage helper has a documented holdout alias limitation; read `docs/EVALUATION.md`.

In [ ]:
# @title Optional offline reproduction
RUN_OFFLINE_REPRODUCTION = False # @param {type:"boolean"}
if RUN_OFFLINE_REPRODUCTION:
    import baseline, evaluate
    for split in ["dev", "holdout"]:
        baseline_dir = baseline.run(f"data/{split}_inputs.json", f"reproduction/{split}_baseline")
        ai_dir = "saved_dev_ai" if split == "dev" else "results/holdout_ai/20261004T075011236089Z"
        summary = evaluate.report(
            f"data/{split}_inputs.json", f"labels/{split}_ground_truth.json",
            {"baseline": str(baseline_dir), "ai": ai_dir},
            f"reproduction/reports/{split}", split, "manual_decisions.json")
        print(split, json.dumps(summary, indent=2, ensure_ascii=False))
else:
    print("Skipped. No baseline or evaluation rerun.")

## 5. Optional: one fresh synthetic demo (may incur API cost)

This new example is outside the scored dataset. It is not part of the final evaluation. Enable only when you want a live demonstration; a private key prompt will appear. Results are kept separately. Do not paste a key into code. No automatic retry is performed.

In [ ]:
# @title Optional paid demonstration
RUN_PAID_DEMO = False # @param {type:"boolean"}
if RUN_PAID_DEMO:
    import getpass, sow
    demo = {"inputs": [{"case_id": "DEMO_NEW_01",
        "declaration": {"doc_id": "DEMO_DECL", "lines": [
            {"line_id": "L1", "text": "Synthetic example: I received a cash gift of SGD 12,000 from my mother on 1 September 2026."}]},
        "documents": [{"doc_id": "DEMO_RECEIPT", "lines": [
            {"line_id": "L1", "text": "Synthetic bank receipt: gift payment SGD 12,000 from the client's mother credited to the client's account on 1 September 2026; completed."}]}]}]}
    Path("demo_results").mkdir(exist_ok=True)
    demo_input = Path("demo_results/demo_input.json")
    demo_input.write_text(json.dumps(demo, ensure_ascii=False, indent=2))
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key (hidden): ").strip()
    try:
        result_dir = sow.run(str(demo_input), "demo_results/runs", case_ids=["DEMO_NEW_01"])
        result = json.loads((result_dir / "DEMO_NEW_01.json").read_text())
        print(json.dumps({k: result.get(k) for k in ["status", "error", "validated_output", "validation", "usage"]}, ensure_ascii=False, indent=2))
    finally:
        os.environ.pop("OPENROUTER_API_KEY", None)
else:
    print("Skipped. No model request and no API key needed.")